# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data\\HealthWellnessGuide.txt', 'data\\MentalHealthGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [12]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [13]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [14]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [16]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Property 'headlines' already exists in node '62a112'. Skipping!
Property 'headlines' already exists in node '7802f3'. Skipping!


Applying HeadlineSplitter:   0%|          | 0/9 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

...

Applying CustomNodeFilter:   0%|          | 0/14 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/30 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '7802f3'. Skipping!
Property 'summary_embedding' already exists in node '62a112'. Skipping!
Property 'themes' already exists in node '67fc1e'. Skipping!
Property 'themes' already exists in node 'f608e9'. Skipping!
Property 'entities' already exists in node '57525a'. Skipping!
Property 'themes' already exists in node '57525a'. Skipping!
Property 'themes' already exists in node '79889f'. Skipping!
Property 'themes' already exists in node 'f04e09'. Skipping!
Property 'themes' already exists in node '3e468e'. Skipping!
Property 'entities' already exists in node '3e468e'. Skipping!
Property 'entities' already exists in node 'f04e09'. Skipping!
Property 'themes' already exists in node '384bfc'. Skipping!
Property 'entities' already exists in node '67fc1e'. Skipping!
Property 'entities' already exists in node '384bfc'. Skipping!
Property 'entities' already exists in node '79889f'. Skipping!
Property 'entities' already exists in node 'f608e9'

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 16, relationships: 42)

We can save and load our knowledge graphs as follows.

In [17]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 16, relationships: 42)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [18]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [19]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
|Synthesizer| Sources needed | Answer style |
|---------|------------|-----------------|
|SingleHopSpecificQuerySynthesizer|One|fact-based, direct|
|MultiHopSpecificQuerySynthesizer|Multiple|fact-based, combined|
|MultiHopAbstractQuerySynthesizer|Multiple|broader, interpretive, summarized|

1. SingleHopSpecificQuerySynthesizer: Generates questions that
    - can be answered from one document or chunk
    - target explicit facts
    - dont need to combine or interpret across sources

2. MultiHopSpecificQuerySynthesizer: Generates questions that
    - require more than one document or chunk
    - ask for facts from each
    - need multiple retrieval steps and then combine all the facts

3. MultiHopAbstractQuerySynthesizer: Generates questions that
    - require more than one document or chunk
    - need interpretation or synthesis instead of copying facts
    - call for comparision, explanation or summarization across sources

Finally, we can use our `TestSetGenerator` to generate our testset!

In [20]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is CBT-I used for in sleep management?,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,"CBT-I, or Cognitive Behavioral Therapy for Ins...",single_hop_specifc_query_synthesizer
1,What are some natural remedies for headaches?,[13: The Science of Habit Formation Habits are...,Natural headache remedies include drinking wat...,single_hop_specifc_query_synthesizer
2,What is the significance of the pelvic in the ...,[The Personal Wellness Guide A Comprehensive R...,The guide mentions pelvic tilts as one of the ...,single_hop_specifc_query_synthesizer
3,How does the Psichology Handbook help with und...,[The Mental Health and Psychology Handbook A P...,The Mental Health and Psychology Handbook is a...,single_hop_specifc_query_synthesizer
4,What is DBT in mental health therapy?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Dialectical Behavior Therapy (DBT) was origina...,single_hop_specifc_query_synthesizer
5,"How do different types of exercise, such as ae...",[<1-hop>\n\nThe Personal Wellness Guide A Comp...,The comprehensive wellness guide emphasizes th...,multi_hop_abstract_query_synthesizer
6,"How do factors influencing mental health, such...",[<1-hop>\n\nThe Mental Health and Psychology H...,Factors influencing mental health include life...,multi_hop_abstract_query_synthesizer
7,How can practicing good sleep hygiene practice...,[<1-hop>\n\n13: The Science of Habit Formation...,"Practicing good sleep hygiene practices, such ...",multi_hop_abstract_query_synthesizer
8,How do Chapters 7 and 16 together inform strat...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Chapter 7 emphasizes the importance of sleep f...,multi_hop_specific_query_synthesizer
9,"How does vitamin D, as a micronutrient discuss...",[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,The provided context highlights that vitamin D...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [21]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [22]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is PART 2 about in the context of healthy...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,"PART 2 covers nutrition and diet, including th...",single_hop_specifc_query_synthesizer
1,What is habit formation?,[13: The Science of Habit Formation Habits are...,Habits are behaviors that become automatic thr...,single_hop_specifc_query_synthesizer
2,Wha is the importnce of Monay in mental health?,[The Personal Wellness Guide A Comprehensive R...,The context does not provide specific informat...,single_hop_specifc_query_synthesizer
3,What role does the World Health Organization p...,[The Mental Health and Psychology Handbook A P...,The context provided does not specify the role...,single_hop_specifc_query_synthesizer
4,How do social connection and boundaries influe...,[<1-hop>\n\nWrite letters to or from your futu...,Strong social connections provide emotional su...,multi_hop_abstract_query_synthesizer
5,How can mindfulness and meditation techniques ...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,The context explains that mindfulness is the p...,multi_hop_abstract_query_synthesizer
6,"How does the mind-body connection, especilly e...",[<1-hop>\n\nThe Mental Health and Psychology H...,The mind-body connection plays a crucial role ...,multi_hop_abstract_query_synthesizer
7,How can developng emotional intelligence throu...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,Developing emotional intelligence involves enh...,multi_hop_abstract_query_synthesizer
8,How can combining Cognitive Behavioral Therapy...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,The context explains that Cognitive Behavioral...,multi_hop_specific_query_synthesizer
9,How do cognitive behavioral therapy (CBT) and ...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,Cognitive Behavioral Therapy (CBT) is a widely...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
|Aspect| Unrolled approach | Abstract approach |
|---------|------------|-----------------|
|Control|You explicitly build the knowledge graph and run the pipeline, hence full control|You use Ragas built-in shortcut, hence limited control|
|Customization|Query types, transforms can be customized|Standard transforms|
|Setup|More code, longer time|Minimal code, faster|

Unrolled appraoch is suitable when

    - we need a custom query distribution example: more multi hop, specific/abstract mix
    - we need domain specific transforms
    - better for production validation

Absract approach can be used when

    - we want to build a fast prototype
    - we are fine with built-in defaults






---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [ ]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
# Generate a new test set and compare with the default

from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

custom_query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),
]

custom_testset = generator.generate(testset_size=10, query_distribution=custom_query_distribution)
custom_testset.to_pandas()

print("Default query distribution (from earlier): single_hop_specific 50%, multi_hop 25% each")
print("Custom query distribution: single_hop 40%, multi_hop 60% combined")


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the main focus of Chapter 4?,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,Chapter 4 covers the fundamentals of healthy e...,single_hop_specifc_query_synthesizer
1,How does understanding the science of habit fo...,[13: The Science of Habit Formation Habits are...,Understanding how habits form can help build p...,single_hop_specifc_query_synthesizer
2,How does the shoulder relate to common issues ...,[The Personal Wellness Guide A Comprehensive R...,The guide mentions that neck and shoulder tens...,single_hop_specifc_query_synthesizer
3,What does the term 'Mental Health' refer to?,[The Mental Health and Psychology Handbook A P...,"Mental health encompasses our emotional, psych...",single_hop_specifc_query_synthesizer
4,How does mental health and well-being relate t...,[<1-hop>\n\nThe Mental Health and Psychology H...,Mental health and well-being encompass our emo...,multi_hop_abstract_query_synthesizer
5,how gut-brain axis and diet affect mental heal...,[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that the gut-brain axis p...,multi_hop_abstract_query_synthesizer
6,Considering the comprehensive insights from th...,[<1-hop>\n\nThe Mental Health and Psychology H...,The handbook explains that physical health dir...,multi_hop_abstract_query_synthesizer
7,H0w can I reliev stress and boost my energy qu...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,To reliev stress and boost your energy quickly...,multi_hop_abstract_query_synthesizer
8,How can incorporating Cognitive Behavioral The...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,Integrating Cognitive Behavioral Therapy (CBT)...,multi_hop_specific_query_synthesizer
9,How can practicing CBT and CBT-I together enha...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,"Practicing Cognitive Behavioral Therapy (CBT),...",multi_hop_specific_query_synthesizer


Reason for why these weights:
I have given a strong focus on multi-hop here 60% vs 40% single hop.
1. MultiHopAbstract at 40% to test whether the system can support understanding and reasoning, not just fact lookup. This is useful for summarization, comparision of use cases.
2. The overall benchmark is harder than a single-hop heavy set.
3. I still want direct fact-based questions, hence 40% single-hop query distribution.

We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [28]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [29]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [30]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [32]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [33]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [34]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [35]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [36]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [37]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [38]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [39]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [40]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [52]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'memorable-discussion-99' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/c1f7c6a2-f757-46e5-a5a8-7b74ba2e3ff3/compare?selectedSessions=e84bcf0f-4f15-4412-90b9-85c28447d464




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,how CBT and CBT-I help mental health and how t...,Based on the context provided:\n\nCBT (Cogniti...,None,Cognitive Behavioral Therapy (CBT) is a widely...,True,True,True,5.710718,98785993-b656-40f2-bb15-195e9436c875,019c6237-488b-7740-96bb-c5f06c919644
1,H0w cbt and cbt-i r rel8ed in menta health str...,Based on the provided context:\n\nCognitive Be...,None,Cognitive Behavioral Therapy (CBT) is a widely...,True,True,True,5.096238,e562ab04-0649-4b07-b3e2-82ec9d481736,019c6237-845b-7533-8eab-a2aac5319715
2,How do cognitive behavioral therapy (CBT) and ...,Cognitive Behavioral Therapy (CBT) contributes...,None,Cognitive Behavioral Therapy (CBT) is a widely...,True,True,True,4.560643,01728d87-6db2-401b-8660-efaf34bd4d0b,019c6237-bf6e-72e2-bcf4-7ca2a4bd9762
3,How can combining Cognitive Behavioral Therapy...,Combining Cognitive Behavioral Therapy (CBT) a...,None,The context explains that Cognitive Behavioral...,True,True,True,3.668245,e9e51623-f310-465c-956e-54e3ce8fa042,019c6238-070a-7503-8d17-25511d9d86e7
4,How can developng emotional intelligence throu...,Developing emotional intelligence through self...,None,Developing emotional intelligence involves enh...,True,True,True,3.906603,d49b7fd8-b7aa-4f0c-9239-2e11ba757387,019c6238-3a2f-7272-8def-067fe9bc0de0
5,"How does the mind-body connection, especilly e...",The mind-body connection shows a powerful inte...,None,The mind-body connection plays a crucial role ...,True,True,True,3.530974,d63a6c35-5f6b-4b70-91a3-1840461b6cc6,019c6238-7905-7b41-a00e-be001146ab49
6,How can mindfulness and meditation techniques ...,"According to the context, mindfulness and medi...",None,The context explains that mindfulness is the p...,True,True,False,2.592218,689a9a6e-b9be-4fd4-88d4-ae67f6c58937,019c6238-afd8-7c90-b964-0ae4d4ff5e61
7,How do social connection and boundaries influe...,Based on the provided context:\n\n**Social con...,None,Strong social connections provide emotional su...,True,True,True,5.538812,fae0740e-74b1-4617-b7e8-dfb8c746601f,019c6238-d97e-7a81-8ef7-037838af2185
8,What role does the World Health Organization p...,I don't know.,None,The context provided does not specify the role...,False,False,False,0.877029,a5798de2-1a29-4011-9261-b4e358d611cf,019c6239-1206-7662-be85-1ee72a144fb7
9,Wha is the importnce of Monay in mental health?,I don't know.,None,The context does not provide specific informat...,False,False,False,0.776374,126d0559-c42e-4323-aca8-ff59c4a3b184,019c6239-35e7-7840-a644-e2dbaa247834


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [42]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [43]:
rag_documents = docs

In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Modifying chunk size changes what gets into the prompt and the context the model uses. Larger chunks can hold more info which can have full concept or may include irrelevant info along side. Whereas smaller chunks can point to narrow & focused info, so less irrelevant text in each chunk but it may not contain the full answer and answers get split across chunks. Hence modifying chunk size changes application performance because it changes what the retriever sees, what the LLM receives and how reasoning unfolds. Therefor it becomes one of the key parameter in RAG system design.

In [45]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Changing the embedding model changes how your chunks and queries are represented in vector space, so it directly affects what gets retrieved and therefore how well the RAG answers. text-embedding-3-large produces higher dimension vector 3072 whereas text-embedding-3-small has 1536 vector dimension, allowing larger embedding model to capture more complex semantic relationships and match complex queries better. However, it comes with higher cost, memory usage and latency.

In [47]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [48]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [49]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [50]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

"Yo, ready to level up your sleep game? Here’s the ultimate recipe to vibe with those sweet, restorative Zzz's straight from the health wisdom archives:\n\n🔥 **Master the Sleep Rhythm:** Lock in a consistent sleep schedule — even on weekends. Your body’s internal clock is a ruthless boss that craves routine.\n\n🔥 **Pre-Sleep Ritual Magic:** Chill out with a relaxing bedtime ritual. Crack open a book, hit gentle stretches, or soak in a warm bath to tell your brain it’s time to power down.\n\n🔥 **Optimize Your Sleep Cave:** Keep your bedroom cool (think 65-68°F / 18-20°C), dark (blackout curtains or stealthy sleep masks), and quiet (white noise machines or earplugs to drown out distractions). Don’t skimp on a killer mattress and pillows — comfort is king.\n\n🔥 **Screen Detox:** Shut down glowing screens at least 1-2 hours before hitting the sack. Blue light is like caffeine in disguise, messing with your melatonin mojo.\n\n🔥 **Mind Your Fuel:** Avoid caffeine after 2 PM and skip heavy me

Finally, we can evaluate the new chain on the same test set!

In [53]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'essential-hall-71' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/c1f7c6a2-f757-46e5-a5a8-7b74ba2e3ff3/compare?selectedSessions=92df5133-bc8e-4944-a459-4237564c98b7




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,how CBT and CBT-I help mental health and how t...,"Alright, here’s the epic lowdown on how CBT an...",None,Cognitive Behavioral Therapy (CBT) is a widely...,True,True,True,7.215892,98785993-b656-40f2-bb15-195e9436c875,019c6247-c0c0-7121-8738-7065265a19d5
1,H0w cbt and cbt-i r rel8ed in menta health str...,"Yo, strap in because here’s the lowdown on how...",None,Cognitive Behavioral Therapy (CBT) is a widely...,True,True,True,6.610673,e562ab04-0649-4b07-b3e2-82ec9d481736,019c6248-0313-7301-92c6-eb2a923a6061
2,How do cognitive behavioral therapy (CBT) and ...,"Alright, let’s dive into this mental health ma...",None,Cognitive Behavioral Therapy (CBT) is a widely...,True,True,True,7.746070,01728d87-6db2-401b-8660-efaf34bd4d0b,019c6248-572b-7e73-8281-52acac44973a
3,How can combining Cognitive Behavioral Therapy...,"Yo, strap in because merging the powerhouse vi...",None,The context explains that Cognitive Behavioral...,True,True,True,5.379645,e9e51623-f310-465c-956e-54e3ce8fa042,019c6248-a595-7632-90d4-bab2130a33dc
4,How can developng emotional intelligence throu...,"Alright, buckle up for some serious mind flexi...",None,Developing emotional intelligence involves enh...,True,True,True,6.972300,d49b7fd8-b7aa-4f0c-9239-2e11ba757387,019c6248-e0ee-7f50-907f-ca7d434f776c
5,"How does the mind-body connection, especilly e...","Yo, let’s get into this mind-body synergy that...",None,The mind-body connection plays a crucial role ...,True,True,True,6.946402,d63a6c35-5f6b-4b70-91a3-1840461b6cc6,019c6249-261a-7670-80e9-0ee547bb396c
6,How can mindfulness and meditation techniques ...,"Oh, buckle up, because mindfulness and meditat...",None,The context explains that mindfulness is the p...,True,True,True,4.135406,689a9a6e-b9be-4fd4-88d4-ae67f6c58937,019c6249-6ccd-7b01-88ff-b7f6834d83d9
7,How do social connection and boundaries influe...,"Alright, buckle up for a mind-blowing synergy ...",None,Strong social connections provide emotional su...,True,True,True,4.913452,fae0740e-74b1-4617-b7e8-dfb8c746601f,019c6249-b74a-7842-aa91-26c118ca58c7
8,What role does the World Health Organization p...,"Yo, the World Health Organization (WHO) is lik...",None,The context provided does not specify the role...,False,False,True,2.475857,a5798de2-1a29-4011-9261-b4e358d611cf,019c624a-07de-7e81-b74e-05905196146b
9,Wha is the importnce of Monay in mental health?,I don't know. The context drops legendary wisd...,None,The context does not provide specific informat...,False,True,True,1.540976,126d0559-c42e-4323-aca8-ff59c4a3b184,019c624a-3686-7751-b8ad-17316183201a


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:
Screenshot for Baseline chain: chunk_size=500, text-embedding-3-small, default RAG prompt.
![Baseline chain screenshot](data/images/baseline_screenshot.png)

Screenshot for Dopeness chain: chunk_size=1000, text-embedding-3-large, dope prompt.
![Baseline chain screenshot](data/images/dopeness_screenshot.png)

Metrics 
|Aspect| Baseline Chain | Dopeness Chain |
|---------|------------|-----------------|
|Chunk size|500|1000|
|Embedding model|text-embedding-3-small|text-embedding-3-large|
|Prompt|default RAG prompt|dope prompt|
|Dopeness (Avg)|0.583|1.0|
|Helpfulness (Avg)|0.75|0.833|
|QA (Avg)|0.75|0.75|
|Latency P50|3.60|5.21|
|Total Tokens|17,526|15,268|
|Input Tokens|15,323|11,420|
|Output Tokens|2203|3848|
|Total cost|$0.0097|$0.0108|
|Input cost|$0.0062|$0.0046|
|Output cost|$0.0036|$0.0062|

1. The dopeness chain uses a prompt that explictly asks for 'rad', 'dope' and 'non-generic' answers. The dopeness evaluator is judging exactly that. Hence dopeness increase from 0.583 to 1.
2. The dopeness chain uses large chunk size of 1000 which can give slightly more complete context and also large embedding model which can retrieve slightly more relevant data. This explains the increase in helpfulness metric.
3. QA is about factual correctness vs the reference. No change in this metric. The test dataset is not sensitive enough to show a difference in this case.
4. For the dopeness chain, with large chunk size and large embedding model the answers are longer as output tokens 3848 > 2203, This pushes the latency up, 3.6 vs 5.21 and also the token output cost.

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores